# Smile Dynamics — Bergomi (2004)
## Implémentation complète sur l'Euro Stoxx 50 (SX5E)

---

> **Référence :** Lorenzo Bergomi, *Smile Dynamics*, Société Générale, April 2004.  
> **Objectif :** Reproduire et étendre l'analyse empirique et théorique du papier sur une fenêtre glissante de **5 ans se terminant à aujourd'hui**, en lieu et place de la période 1999–2004 utilisée dans l'article original.

---

## Plan du notebook

| Section | Contenu |
|---|---|
| **0** | Setup & données de marché (MDX) |
| **1** | Introduction & motivation (Napoleon option) |
| **2** | Cadre de pricing et hedging (Heston & Jump) |
| **3** | Modèle de Heston — propriétés statiques |
| **4** | Calibration du modèle de Heston sur 5 ans |
| **5** | Dynamique des vols implicites — analyse historique |
| **6** | Forward smiles & forward-start options |
| **7** | Delta et comparaison avec Local Vol |
| **8** | Modèles Jump/Lévy — Variance Swaps |
| **9** | Extension stochastique des modèles de Lévy |
| **10** | Synthèse & conclusions |


---
## Section 0 — Setup & données de marché

In [ ]:
# ============================================================
#  IMPORTS
# ============================================================
import datetime as dt
import warnings
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import PercentFormatter
from scipy.stats import norm, chi2
from scipy.optimize import minimize, brentq, differential_evolution
from scipy.integrate import quad
from pandas.tseries.offsets import BDay

warnings.simplefilter('ignore')

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})

print('Imports OK')

In [ ]:
# ============================================================
#  CONNEXION MDX — RENSEIGNER ICI LES IDENTIFIANTS
# ============================================================
import ezmdx
from maxxpy.apis.mdx.api import MdxClient

LOGIN_MDX    = ""        # <-- TON LOGIN
PASSWORD_MDX = ""        # <-- TON MOT DE PASSE

MDX_TYPES = {
    'volatility': 'EQUITY_VOLATILITY',
    'spot': ['STOCK_QUOTE', 'INDEX_QUOTE', 'FUND_QUOTE']
}

EUROSTOXX_MDX_CODE = 'STOX5E_X'

ezmdx.set_app(app_name='VEGA5')
ezmdx.prod.satis_login()

mtx_client = MdxClient(
    'MSD',
    LOGIN_MDX,
    PASSWORD_MDX,
    use_prod_only=True
)

In [ ]:
# ============================================================
#  PLAGE DE DATES — 5 ans glissants jusqu'à hier
# ============================================================
today      = pd.Timestamp.today().normalize()
date_end   = today - BDay(1)
date_start = date_end - pd.DateOffset(years=5)

print(f'Période d\'analyse : {date_start.date()}  →  {date_end.date()}')
print(f'(analogue à la période 1999–2004 du papier Bergomi)')

In [ ]:
# ============================================================
#  FONCTIONS DE RÉCUPÉRATION DES DONNÉES
# ============================================================
def get_market_data(mtx_client, asset_name, date_range, mdx_type):
    query = {'mdx_type': mdx_type, 'code': asset_name, 'date': date_range}
    return mtx_client.get_market_data(**query)


def get_vol(asset_name, asset_type, date_start, date_end, mtx_client):
    all_bdays = pd.bdate_range(date_start, date_end).strftime('%Y-%m-%d').tolist()
    code_add_on = f'{asset_type}_{asset_name}'
    df = get_market_data(mtx_client, code_add_on, all_bdays, MDX_TYPES['volatility'])
    return df[['STRIKE', 'MATURITY', 'VOLATILITY', 'DATE']].copy()


def get_spot(mtx_client, asset_name, date_start, date_end):
    all_bdays = pd.bdate_range(date_start, date_end).strftime('%Y-%m-%d').tolist()
    for mdx_type in MDX_TYPES['spot']:
        try:
            return get_market_data(mtx_client, asset_name, all_bdays, mdx_type)
        except Exception:
            continue
    return None


# ============================================================
#  CHARGEMENT (avec cache pour éviter les re-fetch)
# ============================================================
cache_dir  = Path('./cache')
cache_dir.mkdir(exist_ok=True)
cache_path = cache_dir / 'sx5e_bergomi_5y_cache.pkl'

if cache_path.exists():
    print('Cache trouvé, chargement...')
    with open(cache_path, 'rb') as f:
        cached = pickle.load(f)
    vols_raw  = cached['vols']
    spots_raw = cached['spots']
else:
    print('Fetching SX5E data depuis MDX...')
    vols_raw  = get_vol(EUROSTOXX_MDX_CODE, 'I', date_start, date_end, mtx_client)
    spots_raw = get_spot(mtx_client, EUROSTOXX_MDX_CODE,
                         date_start - BDay(5), date_end + BDay(5))
    with open(cache_path, 'wb') as f:
        pickle.dump({'vols': vols_raw, 'spots': spots_raw}, f)
    print('Cache sauvegardé :', cache_path)

print(f'Vols : {vols_raw.shape}   |   Spots : {spots_raw.shape}')

In [ ]:
# ============================================================
#  CONSTRUCTION DE LA SURFACE
# ============================================================
def business_days_between(mat_date, ref_date):
    return len(pd.bdate_range(start=ref_date, end=mat_date)) - 1


def build_surface(vols_raw, spots_raw, r=0.0, q=0.0):
    vols = vols_raw.rename(columns={
        'STRIKE': 'strike', 'MATURITY': 'maturity_date',
        'VOLATILITY': 'market_iv', 'DATE': 'date'
    })
    vols['date']         = pd.to_datetime(vols['date'])
    vols['maturity_date'] = pd.to_datetime(vols['maturity_date'])
    vols['strike']       = pd.to_numeric(vols['strike'],    errors='coerce')
    vols['market_iv']    = pd.to_numeric(vols['market_iv'], errors='coerce')
    if vols['market_iv'].median() > 2:
        vols['market_iv'] /= 100.0

    spots = spots_raw.copy()
    spots['date'] = pd.to_datetime(spots['DATE'])
    spot_col = [c for c in spots.columns if c != 'DATE'][0]
    spots['spot'] = pd.to_numeric(spots[spot_col], errors='coerce')
    spots = spots[['date', 'spot']]

    df = vols.merge(spots, on='date', how='left').dropna()
    df['bdays_to_mat'] = [
        business_days_between(m, d)
        for m, d in zip(df['maturity_date'], df['date'])
    ]
    df = df[df['bdays_to_mat'] > 0]
    df['T']           = df['bdays_to_mat'] / 252.0
    df['forward']     = df['spot'] * np.exp((r - q) * df['T'])
    df['log_moneyness'] = np.log(df['strike'] / df['forward'])

    return df.sort_values(['date', 'T', 'strike']).reset_index(drop=True)


surface = build_surface(vols_raw, spots_raw)
print(f'Surface construite : {surface.shape[0]:,} points')
surface.head(10)

---
## Section 1 — Introduction & motivation : la Napoleon option

### 1.1 Contexte

Dans l'approche classique Black-Scholes, les volatilités implicites sont (a) identiques pour tous les strikes et (b) figées dans le temps. Les modèles de smile cherchent à corriger (a). Bergomi souligne que pour les structures exotiques récentes — **Napoleons, reverse cliquets** — c'est l'hypothèse (b) qui est critique.

Pour une telle option, les grecques croisées $\dfrac{\partial^2 P}{\partial \hat{\sigma}^2}$ et $\dfrac{\partial^2 P}{\partial S \, \partial \hat{\sigma}}$ sont significatives. Un modèle correct doit **pricer un Theta** qui compense ces Gammas croisés.

### 1.2 Payoff de la Napoleon

Le coupon annuel en fin d'année $k$ vaut :
$$C_k = \max\!\left(8\% + \min_{j=1,\dots,12} r_{k,j},\; 0\right)$$
où $r_{k,j}$ est la performance mensuelle du SX5E pour le $j$-ème mois de l'année $k$.

La Napoleon est en substance une **Put sur la volatilité forward longue (1 an)**.

In [ ]:
# ============================================================
#  Figure 1.1 (réplique) : valeur Napoleon en fonction de la vol
# ============================================================
def napoleon_coupon_bs(S0, vol, T=1.0, n_monthly=12, coupon=0.08, n_mc=100_000, seed=42):
    """Prix MC Black-Scholes d'un coupon Napoleon de maturité T."""
    rng  = np.random.default_rng(seed)
    dt   = T / n_monthly
    Z    = rng.standard_normal((n_mc, n_monthly))
    log_r = (- 0.5 * vol**2 * dt) + vol * np.sqrt(dt) * Z
    monthly_rets = np.exp(log_r) - 1          # shape (n_mc, n_monthly)
    worst = monthly_rets.min(axis=1)           # worst of 12 monthly returns
    payoff = np.maximum(coupon + worst, 0.0)
    return payoff.mean()


vols_grid = np.linspace(0.05, 0.45, 40)
napoleon_values = [napoleon_coupon_bs(100, v) for v in vols_grid]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gauche : valeur Napoleon vs volatilité
axes[0].plot(vols_grid * 100, np.array(napoleon_values) * 100, 'steelblue', lw=2)
axes[0].set_xlabel('Volatilité (%)')
axes[0].set_ylabel('Valeur (% notionnel)')
axes[0].set_title('Valeur Napoleon (coupons an 3-6) vs Volatilité')
axes[0].yaxis.set_major_formatter(PercentFormatter(xmax=100))

# Droite : Vega du coupon an 3, mois 1 vs spot
spots_grid = np.linspace(84, 104, 50)
eps = 0.01

def coupon_vega_vs_spot(spots, vol=0.20, T=1.0, n_mc=50_000):
    """Vega approximée par différences finies."""
    vegas = []
    for s in spots:
        vu = napoleon_coupon_bs(s, vol + eps, T=T, n_mc=n_mc)
        vd = napoleon_coupon_bs(s, vol - eps, T=T, n_mc=n_mc)
        vegas.append((vu - vd) / (2 * eps))
    return np.array(vegas)

vegas = coupon_vega_vs_spot(spots_grid)
axes[1].plot(spots_grid, vegas * 100, 'firebrick', lw=2)
axes[1].axhline(0, color='k', lw=0.8, ls='--')
axes[1].set_xlabel('Spot (base 100 en début d\'année)')
axes[1].set_ylabel('Vega (% notionnel / point de vol)')
axes[1].set_title('Vega du coupon (1er mois an 3) vs Spot  |  σ = 20%')

plt.suptitle('Figure 1.1 — Réplique Bergomi (2004)', fontweight='bold')
plt.tight_layout()
plt.show()

**Lecture :** Le Vega est décroissant avec le spot et s'annule pour les faibles valeurs (le coupon vaut zéro). Quand le spot baisse, le vendeur doit racheter du Vega, mais les baisses de spot sont **négativement corrélées** avec les hausse de vol implicite → P&L négatif non pricé en Black-Scholes.

---
## Section 2 — Cadre de pricing et hedging

### 2.1 Principe général

Bergomi minimise la **variance du P&L final actualisé** du hedger :

$$\text{P\&L} = -e^{-r(T-t)} f(S_T) + \int_t^T e^{-r(\tau-t)} \Delta(\tau, S, \ldots)\bigl(dS_\tau - (r-q)S\,d\tau\bigr)$$

Le **prix** est défini comme $P = -\mathbb{E}[\text{P\&L}]$.

### 2.2 Modèle de Heston

Dynamique historique :
$$dS = \mu S\,dt + \sqrt{V}\,S\,dZ_t$$
$$dV = -k(V - V_0)\,dt + \sigma\sqrt{V}\,dW_t$$
avec $d\langle Z, W\rangle_t = \rho\,dt$.

L'équation de pricing résultante est :
$$\boxed{\frac{\partial P}{\partial t} + (r-q)S\frac{\partial P}{\partial S} - k(V-V_0)\frac{\partial P}{\partial V} + \frac{1}{2}VS^2\frac{\partial^2 P}{\partial S^2} + \frac{1}{2}\sigma^2 V\frac{\partial^2 P}{\partial V^2} + \rho\sigma SV\frac{\partial^2 P}{\partial S\,\partial V} = rP}$$

avec le **Delta optimal** :
$$\Delta = \frac{\partial P}{\partial S} + \rho\sigma S\frac{\partial P}{\partial V}$$

Le second terme est le ratio covariance(V, S) / variance(S) — le **vanna hedge**.

### 2.3 Modèles Jump-diffusion (Merton)

$$dS = \mu S\,dt + \sigma S\,dZ_t + JS\,dq_t$$

En choisissant $\Delta = \partial P / \partial S$ (BS delta — on évite la mise sur le drift historique) :

$$\boxed{\frac{\partial P}{\partial t} + (r-q)S\frac{\partial P}{\partial S} + \lambda(\delta P - JS\frac{\partial P}{\partial S}) + \frac{\sigma^2 S^2}{2}\frac{\partial^2 P}{\partial S^2} = rP}$$

où $\delta P = P(S(1+J), t) - P(S, t)$.

---
## Section 3 — Modèle de Heston : propriétés statiques

### 3.1 Formule de Heston (1993) — transformée de Fourier

Le prix d'un call européen dans le modèle de Heston s'écrit :
$$C = S_0 e^{-qT} P_1 - K e^{-rT} P_2$$
où $P_j$ s'obtient via la transformée de Fourier de la fonction caractéristique $\phi_j(u)$ :
$$P_j = \frac{1}{2} + \frac{1}{\pi} \int_0^\infty \mathrm{Re}\!\left[\frac{e^{-iu\ln K} \phi_j(u)}{iu}\right] du$$

Avec la fonction caractéristique :
$$\phi_j(u) = \exp\!\left(C_j(u,T) + D_j(u,T)\,V_0 + iu\ln F\right)$$

### 3.2 Développements perturbatifs (Bergomi eq. 3.1 & 3.2)

**Court terme** ($T \ll \tau = 1/k$) :
$$\hat{\sigma}_F \approx \sqrt{V}, \qquad \left.\frac{d\hat{\sigma}}{d\ln K}\right|_F \approx \frac{\rho\sigma}{4\sqrt{V}}$$

**Long terme** ($T \gg \tau$) :
$$\hat{\sigma}_F \approx \sqrt{V_0}\left(1 + \frac{\rho\sigma}{4k}\right) + \frac{\sqrt{V_0}}{2kT}\!\left(\frac{V-V_0}{V_0} + \frac{\rho\sigma}{4k}\frac{V - 3V_0}{V_0}\right)$$
$$\left.\frac{d\hat{\sigma}}{d\ln K}\right|_F \approx \frac{\rho\sigma}{2kT\sqrt{V_0}}$$

**Variance Swap volatility** :
$$\hat{\sigma}_{VS}^2(T) = V_0 + (V - V_0)\frac{1 - e^{-kT}}{kT}$$

In [ ]:
# ============================================================
#  FORMULE DE HESTON — PRICING PAR TRANSFORMÉE DE FOURIER
# ============================================================
def heston_char_func(u, S0, V, V0, kappa, sigma, rho, r, q, T, j):
    """Fonction caractéristique de Heston (formulation numérique stable)."""
    F   = S0 * np.exp((r - q) * T)
    x   = np.log(F)
    b_j = kappa - (rho * sigma * j)  # j=1 ou 0 selon P1 ou P2
    u_j = u - 1j * j                 # shift pour P1
    # — version Albrecher & al. (2007) pour éviter les coupures de branche —
    d   = np.sqrt((rho * sigma * u_j * 1j - b_j)**2 +
                   sigma**2 * (u_j * 1j + u_j**2))
    g   = (b_j - rho * sigma * u_j * 1j + d) / \
          (b_j - rho * sigma * u_j * 1j - d)
    C   = kappa * (V0 / sigma**2) * (
          (b_j - rho * sigma * u_j * 1j + d) * T
          - 2 * np.log((1 - g * np.exp(d * T)) / (1 - g))
          )
    D   = ((b_j - rho * sigma * u_j * 1j + d) / sigma**2) * \
          ((1 - np.exp(d * T)) / (1 - g * np.exp(d * T)))
    return np.exp(C + D * V + 1j * u * x)


def heston_price(S0, K, T, V, V0, kappa, sigma, rho, r=0.0, q=0.0,
                 n_quad=200):
    """Prix d'un call européen par quadrature de Gauss-Laguerre."""
    def integrand(u, j):
        phi = heston_char_func(u, S0, V, V0, kappa, sigma, rho, r, q, T, j)
        return np.real(np.exp(-1j * u * np.log(K)) * phi / (1j * u))

    P1, _ = quad(lambda u: integrand(u, 1), 1e-6, 500, limit=n_quad)
    P2, _ = quad(lambda u: integrand(u, 0), 1e-6, 500, limit=n_quad)
    P1 = 0.5 + P1 / np.pi
    P2 = 0.5 + P2 / np.pi
    return S0 * np.exp(-q * T) * P1 - K * np.exp(-r * T) * P2


def bs_price(S, K, T, sigma, r=0.0, q=0.0, option='call'):
    """Prix Black-Scholes d'un call ou put."""
    F  = S * np.exp((r - q) * T)
    d1 = (np.log(F / K) + 0.5 * sigma**2 * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    discount = np.exp(-r * T)
    if option == 'call':
        return discount * (F * norm.cdf(d1) - K * norm.cdf(d2))
    else:
        return discount * (K * norm.cdf(-d2) - F * norm.cdf(-d1))


def implied_vol(S, K, T, price, r=0.0, q=0.0, option='call', tol=1e-8):
    """Vol implicite Black-Scholes par méthode de Brent."""
    intrinsic = max(S * np.exp(-q * T) - K * np.exp(-r * T), 0.0)
    if price <= intrinsic + 1e-10:
        return np.nan
    try:
        return brentq(
            lambda v: bs_price(S, K, T, v, r, q, option) - price,
            1e-4, 5.0, xtol=tol
        )
    except Exception:
        return np.nan


print('Fonctions de pricing Heston & BS définies.')

In [ ]:
# ============================================================
#  SMILE DE HESTON — ILLUSTRATION AVEC PARAMÈTRES TYPIQUES
# ============================================================
# Paramètres typiques du papier Bergomi
PARAMS_REF = dict(V=0.1, V0=0.1, kappa=2.0, sigma=1.0, rho=-0.7)

S0 = 100.0
maturities_ref = [0.25, 0.5, 1.0, 2.0]
log_m_grid = np.linspace(-0.4, 0.4, 60)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for T in maturities_ref:
    F = S0  # r=q=0
    ivs = []
    for lm in log_m_grid:
        K   = F * np.exp(lm)
        opt = 'put' if K < F else 'call'
        px  = heston_price(S0, K, T, **PARAMS_REF)
        if opt == 'put':
            px = px - S0 + K  # put-call parity
            px = max(px, 1e-10)
        iv = implied_vol(S0, K, T, px, option='call')
        ivs.append(iv)
    axes[0].plot(log_m_grid * 100, np.array(ivs) * 100,
                 label=f'T = {T}y')

axes[0].set_xlabel('Log-moneyness ln(K/F)  (%)')
axes[0].set_ylabel('Vol implicite (%)')
axes[0].set_title('Smile Heston — paramètres de référence Bergomi')
axes[0].legend()

# Variance Swap vol vs T
T_grid  = np.linspace(0.05, 3.0, 100)
V, V0, k = PARAMS_REF['V'], PARAMS_REF['V0'], PARAMS_REF['kappa']
vs_vol = np.sqrt(V0 + (V - V0) * (1 - np.exp(-k * T_grid)) / (k * T_grid))
axes[1].plot(T_grid, vs_vol * 100, 'steelblue', lw=2, label='$\\hat{\\sigma}_{VS}(T)$')
axes[1].axhline(np.sqrt(V0) * 100, ls='--', color='gray', label='$\\sqrt{V_0}$ (long terme)')
axes[1].axhline(np.sqrt(V)  * 100, ls=':', color='firebrick', label='$\\sqrt{V}$ (court terme)')
axes[1].set_xlabel('Maturité T (années)')
axes[1].set_ylabel('$\\hat{\\sigma}_{VS}$ (%)')
axes[1].set_title('Structure par terme de la Variance Swap Vol')
axes[1].legend()

plt.suptitle('Figure 3 — Propriétés statiques du modèle de Heston', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
#  APPROXIMATIONS PERTURBATIVES — SKEW & ATMF VOL
# ============================================================
def heston_approx_short(V, kappa, sigma, rho):
    """Éq. 3.1 — court terme T << 1/k."""
    atmf_vol = np.sqrt(V)
    skew     = rho * sigma / (4 * np.sqrt(V))
    return atmf_vol, skew


def heston_approx_long(V, V0, kappa, sigma, rho, T):
    """Éq. 3.2 — long terme T >> 1/k."""
    sq_V0  = np.sqrt(V0)
    atmf   = sq_V0 * (1 + rho * sigma / (4 * kappa)) + \
             sq_V0 / (2 * kappa * T) * (
                 (V - V0) / V0 + rho * sigma / (4 * kappa) * (V - 3 * V0) / V0
             )
    skew   = rho * sigma / (2 * kappa * T * sq_V0)
    return atmf, skew


# Tableau de synthèse des approximations
T_vals = [0.083, 0.25, 0.5, 1.0, 2.0]
rows   = []
for T in T_vals:
    v_short, s_short = heston_approx_short(**{k: PARAMS_REF[k] for k in ['V', 'kappa', 'sigma', 'rho']})
    v_long,  s_long  = heston_approx_long(T=T, **PARAMS_REF)
    rows.append({
        'T (années)': T,
        'ATMF vol (court terme)': f'{v_short*100:.2f}%',
        'Skew (court terme)': f'{s_short*100:.4f}% / ln-strike',
        'ATMF vol (long terme)': f'{v_long*100:.2f}%',
        'Skew (long terme)': f'{s_long*100:.4f}% / ln-strike',
    })

pd.DataFrame(rows).set_index('T (années)')

---
## Section 4 — Calibration du modèle de Heston sur 5 ans (SX5E)

Pour chaque date de business day, on calibre les 4 paramètres libres $(V, V_0, \sigma, \rho)$ avec $k=2$ fixé (comme dans l'article), en minimisant l'erreur quadratique sur les vols implicites de marché.

**Fonction objectif :**
$$\mathcal{L}(V, V_0, \sigma, \rho) = \sum_{i} w_i \left(\hat{\sigma}_i^{\text{model}} - \hat{\sigma}_i^{\text{market}}\right)^2$$

In [ ]:
# ============================================================
#  CALIBRATION HESTON — FONCTION CŒUR
# ============================================================
KAPPA_FIXED = 2.0
R, Q = 0.0, 0.0

def heston_iv_surface(params, S0, strikes, maturities, r=0.0, q=0.0):
    """Calcule les vols implicites Heston pour un vecteur (K, T)."""
    V, V0, sigma, rho = params
    ivs = []
    for K, T in zip(strikes, maturities):
        px = heston_price(S0, K, T, V, V0, KAPPA_FIXED, sigma, rho, r, q)
        opt = 'call' if K >= S0 * np.exp((r - q) * T) else 'put'
        if opt == 'put':
            px_call = px + S0 * np.exp(-q * T) - K * np.exp(-r * T)
            iv = implied_vol(S0, K, T, max(px_call, 1e-10), r, q, 'call')
        else:
            iv = implied_vol(S0, K, T, max(px, 1e-10), r, q, 'call')
        ivs.append(iv if iv is not None else np.nan)
    return np.array(ivs)


def calibrate_heston_day(df_day, S0, r=0.0, q=0.0):
    """
    Calibre Heston sur une date donnée.
    Retourne (V, V0, sigma, rho, rmse) ou None si échec.
    """
    sub = df_day.dropna(subset=['market_iv'])
    if len(sub) < 5:
        return None

    strikes  = sub['strike'].values
    mats     = sub['T'].values
    iv_mkt   = sub['market_iv'].values

    def objective(params):
        V, V0, sigma, rho = params
        ivs = heston_iv_surface(params, S0, strikes, mats, r, q)
        mask = np.isfinite(ivs)
        if mask.sum() < 3:
            return 1e6
        return np.mean((ivs[mask] - iv_mkt[mask])**2)

    # Bornes physiques
    bounds = [
        (1e-4, 2.0),   # V
        (1e-4, 2.0),   # V0
        (0.01, 5.0),   # sigma
        (-0.99, -0.01) # rho (équité → négatif)
    ]
    x0 = [0.04, 0.04, 0.8, -0.5]
    res = minimize(objective, x0, method='L-BFGS-B', bounds=bounds,
                   options={'maxiter': 300, 'ftol': 1e-9})
    if res.success or res.fun < 1e-4:
        V, V0, sigma, rho = res.x
        rmse = np.sqrt(res.fun)
        return V, V0, sigma, rho, rmse
    return None


print('Fonctions de calibration définies.')

In [ ]:
# ============================================================
#  BOUCLE DE CALIBRATION SUR 5 ANS
#  (utilise un sous-cache pour ne pas recalculer)
# ============================================================
calib_cache = cache_dir / 'sx5e_heston_calib.pkl'

if calib_cache.exists():
    print('Cache de calibration trouvé, chargement...')
    with open(calib_cache, 'rb') as f:
        calib_df = pickle.load(f)
else:
    print('Calibration Heston en cours (peut prendre plusieurs minutes)...')
    dates = surface['date'].unique()
    results = []

    for i, d in enumerate(dates):
        day_df = surface[surface['date'] == d]
        spot   = day_df['spot'].iloc[0]
        res    = calibrate_heston_day(day_df, S0=spot, r=R, q=Q)
        if res is not None:
            V, V0, sigma, rho, rmse = res
            results.append({
                'date': d, 'V': V, 'V0': V0,
                'sigma': sigma, 'rho': rho, 'rmse': rmse
            })
        if (i + 1) % 50 == 0:
            print(f'  {i+1}/{len(dates)} dates traitées')

    calib_df = pd.DataFrame(results).set_index('date')
    calib_df.index = pd.to_datetime(calib_df.index)
    calib_df = calib_df.sort_index()

    with open(calib_cache, 'wb') as f:
        pickle.dump(calib_df, f)
    print('Calibration terminée et mise en cache.')

print(f'\n{len(calib_df)} dates calibrées sur {len(surface["date"].unique())} disponibles')
calib_df.describe().round(4)

In [ ]:
# ============================================================
#  FIGURE 3.1 — RÉPLIQUE : paramètres calibrés dans le temps
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

params_plot = [
    ('V',     'V (variance instantanée)',  'steelblue'),
    ('V0',    'V₀ (variance long terme)',  'darkorange'),
    ('sigma', 'σ (vol-de-vol)',            'forestgreen'),
    ('rho',   'ρ (corrélation spot/vol)',  'firebrick'),
]

for ax, (col, label, color) in zip(axes, params_plot):
    ax.plot(calib_df.index, calib_df[col], color=color, lw=1.2, alpha=0.9)
    ax.set_ylabel(label)
    ax.set_title(label)
    ax.xaxis.set_tick_params(rotation=30)

# Superposer V et sigma (×1/10) comme Bergomi
ax_sigma = axes[2]
ax2      = ax_sigma.twinx()
ax2.plot(calib_df.index, calib_df['V'], color='steelblue',
         lw=1, alpha=0.5, ls='--', label='V (axe droit)')
ax2.set_ylabel('V', color='steelblue')
ax_sigma.set_title('σ (vol-de-vol)  — V superposé en tirets')

plt.suptitle('Figure 3.1 — Paramètres Heston calibrés sur 5 ans (SX5E)', fontweight='bold')
plt.tight_layout()
plt.show()

### 3.3 Observation clé de Bergomi

D'après l'équation 3.1, le skew court terme Heston vaut $\rho\sigma / (4\sqrt{V})$. Si les skews de marché sont **proportionnels à la vol ATM** (et non inversement proportionnels), on devrait observer que $\sigma \propto V$, c'est-à-dire $\sigma / V \approx \text{const}$.

In [ ]:
# ============================================================
#  TEST : corrélation sigma ~ V  (Bergomi section 3.2.1)
# ============================================================
corr_sigma_V = calib_df[['sigma', 'V']].corr().loc['sigma', 'V']
print(f'Corrélation(σ, V) = {corr_sigma_V:.3f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(calib_df['V'], calib_df['sigma'],
                alpha=0.3, s=15, color='steelblue')
axes[0].set_xlabel('V (variance instantanée)')
axes[0].set_ylabel('σ (vol-de-vol)')
axes[0].set_title(f'Corrélation(σ, V) = {corr_sigma_V:.3f}')

# Ratio sigma/V dans le temps
ratio = calib_df['sigma'] / calib_df['V']
axes[1].plot(calib_df.index, ratio, color='darkorange', lw=1.2, alpha=0.8)
axes[1].axhline(ratio.median(), ls='--', color='k', label=f'Médiane = {ratio.median():.1f}')
axes[1].set_ylabel('σ / V')
axes[1].set_title('Ratio σ/V dans le temps')
axes[1].legend()
axes[1].xaxis.set_tick_params(rotation=30)

plt.suptitle('Bergomi section 3.2.1 — σ corrélée avec V', fontweight='bold')
plt.tight_layout()
plt.show()

---
## Section 5 — Dynamique des vols implicites : analyse historique

### 5.1 Extraction des vols ATM par maturité

Bergomi extrait les vols ATM pour les maturités **1 mois, 3 mois, 6 mois, 1 an** et les compare aux prédictions du modèle.

Il calcule ensuite les ratios :
$$R_S = \left\langle \frac{(\delta S)^2}{S^2 V \delta t} \right\rangle \qquad R_V = \left\langle \frac{(\delta V)^2}{\sigma^2 V \delta t} \right\rangle \qquad R_{SV} = \left\langle \frac{\delta S \cdot \delta V}{\rho \sigma S V \delta t} \right\rangle$$

qui mesurent dans quelle mesure le modèle est **consistant avec la dynamique historique**. La valeur théorique est 1 pour les trois ratios.

In [ ]:
# ============================================================
#  EXTRACTION VOLS ATM PAR MATURITÉ
# ============================================================
# Cibles de maturité en années
TARGET_MATS = {"1M": 1/12, "3M": 3/12, "6M": 6/12, "1Y": 1.0}
TOLERANCE   = 0.02  # +/- 2 semaines

atm_vols = {}

for label, T_target in TARGET_MATS.items():
    rows = []
    for date, grp in surface.groupby('date'):
        # Sélection de la maturité la plus proche
        avail_T = grp['T'].unique()
        closest = avail_T[np.argmin(np.abs(avail_T - T_target))]
        if abs(closest - T_target) > TOLERANCE:
            continue
        slice_T = grp[np.abs(grp['T'] - closest) < 1e-3]
        # Option la plus proche du forward (log_moneyness ~ 0)
        idx_atm = slice_T['log_moneyness'].abs().idxmin()
        rows.append({
            'date': date,
            'atm_iv': slice_T.loc[idx_atm, 'market_iv'],
            'T_actual': closest,
            'spot': slice_T.loc[idx_atm, 'spot'],
        })
    atm_vols[label] = pd.DataFrame(rows).set_index('date').sort_index()
    atm_vols[label].index = pd.to_datetime(atm_vols[label].index)

print('Vols ATM extraites :')
for k, v in atm_vols.items():
    print(f'  {k}: {len(v)} dates')

In [ ]:
# ============================================================
#  FIGURE 3.2 — VOL ATM vs PROXY MODÈLE
# ============================================================
# Proxy court terme : sqrt(V) — proxy long terme : sigma_VS(1Y)
calib_aligned = calib_df.reindex(atm_vols['1M'].index, method='nearest')
vs_1y = np.sqrt(
    calib_aligned['V0'] +
    (calib_aligned['V'] - calib_aligned['V0']) *
    (1 - np.exp(-KAPPA_FIXED * 1.0)) / (KAPPA_FIXED * 1.0)
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Gauche : 1M ATM vol vs sqrt(V)
common_1m = atm_vols['1M'].join(calib_df[['V']], how='inner')
axes[0].plot(common_1m.index, common_1m['atm_iv'] * 100, 'steelblue',
             lw=1.2, label='ATM 1M (marché)')
axes[0].plot(common_1m.index, np.sqrt(common_1m['V']) * 100, 'firebrick',
             lw=1.2, ls='--', label='$\\sqrt{V}$ (modèle)')
axes[0].set_ylabel('Volatilité (%)')
axes[0].set_title('Vol ATM 1 mois')
axes[0].legend()
axes[0].xaxis.set_tick_params(rotation=30)

# Droite : 1Y ATM vol vs sigma_VS(1Y)
common_1y = atm_vols['1Y'].join(calib_df[['V', 'V0']], how='inner')
vs_1y_aligned = np.sqrt(
    common_1y['V0'] +
    (common_1y['V'] - common_1y['V0']) *
    (1 - np.exp(-KAPPA_FIXED)) / KAPPA_FIXED
)
axes[1].plot(common_1y.index, common_1y['atm_iv'] * 100, 'steelblue',
             lw=1.2, label='ATM 1Y (marché)')
axes[1].plot(common_1y.index, vs_1y_aligned * 100, 'firebrick',
             lw=1.2, ls='--', label='$\\hat{\\sigma}_{VS}(1Y)$ (modèle)')
axes[1].set_ylabel('Volatilité (%)')
axes[1].set_title('Vol ATM 1 an')
axes[1].legend()
axes[1].xaxis.set_tick_params(rotation=30)

plt.suptitle('Figure 3.2 — Suivi des vols ATM par le modèle Heston (SX5E, 5 ans)',
             fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
#  RATIOS R_S, R_V, R_SV  — CONSISTANCE DU MODÈLE
#  (Bergomi section 3.2, p. 7)
# ============================================================
# On a besoin de : spot, V, sigma, rho — alignés sur les mêmes dates
spot_series = atm_vols['1M'][['spot']].copy()
joint = spot_series.join(calib_df[['V', 'sigma', 'rho']], how='inner').dropna()
joint = joint.sort_index()

dt_years = 1 / 252.0  # pas journalier

dS_S = joint['spot'].pct_change().dropna()
dV   = joint['V'].diff().dropna()

common_idx = dS_S.index.intersection(dV.index)
dS_S = dS_S.loc[common_idx]
dV   = dV.loc[common_idx]
V_c  = joint.loc[common_idx, 'V']
sig_c = joint.loc[common_idx, 'sigma']
rho_c = joint.loc[common_idx, 'rho']

R_S  = np.mean(dS_S**2 / (V_c * dt_years))
R_V  = np.mean(dV**2 / (sig_c**2 * V_c * dt_years))
R_SV = np.mean(dS_S * dV / (rho_c * sig_c * V_c * dt_years))

print('=' * 55)
print('  Ratios de consistance du modèle Heston (global 5 ans)')
print('  (Valeur théorique = 1.0 si le modèle est exact)\n')
print(f'  R_S  = {R_S:.3f}   (variance spot)  — Bergomi : 0.75')
print(f'  R_V  = {R_V:.3f}   (variance vol)   — Bergomi : 0.40')
print(f'  R_SV = {R_SV:.3f}  (covariance)     — Bergomi : 0.60')
print()
sr = R_SV / (np.sqrt(R_V * R_S))
print(f'  σ_réalisée / σ_implicite  = {np.sqrt(R_V):.3f}')
print(f'  ρ_réalisée / ρ_implicite  = {sr:.3f}')
print('=' * 55)

In [ ]:
# ============================================================
#  FIGURE 3.3 — MOYENNES GLISSANTES MENSUELLES
# ============================================================
roll_window = 21  # ~1 mois

v_real_dS = (dS_S**2).rolling(roll_window).mean()
v_impl_dS = (V_c * dt_years).rolling(roll_window).mean()

v_real_dV  = (dV**2).rolling(roll_window).mean()
v_impl_dV  = (sig_c**2 * V_c * dt_years).rolling(roll_window).mean()

c_real  = (dS_S * dV).rolling(roll_window).mean()
c_impl  = (rho_c * sig_c * V_c * dt_years).rolling(roll_window).mean()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (real, impl, title) in zip(axes, [
    (v_real_dS, v_impl_dS, '$V_{\\delta S/S}^{\\text{réel}}$ vs $V_{\\delta S/S}^{\\text{impl}}$'),
    (v_real_dV, v_impl_dV, '$V_{\\delta V}^{\\text{réel}}$ vs $V_{\\delta V}^{\\text{impl}}$'),
    (c_real,    c_impl,    '$C_{\\delta S/S,\\delta V}^{\\text{réel}}$ vs $C^{\\text{impl}}$'),
]):
    ax.plot(real.index, real.values, 'steelblue', lw=1.2, label='Réalisé')
    ax.plot(impl.index, impl.values, 'firebrick', lw=1.2, ls='--', label='Implicite (modèle)')
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.xaxis.set_tick_params(rotation=30)

plt.suptitle('Figure 3.3 — Moyennes glissantes : quantités réalisées vs implicites',
             fontweight='bold')
plt.tight_layout()
plt.show()

### 5.2 Interprétation

| Ratio | Formule | Valeur théorique | Bergomi (1999-2004) | Notre estimation (5 ans) |
|---|---|---|---|---|
| $R_S$ | $\langle (\delta S)^2 / (V S^2 \delta t) \rangle$ | 1.0 | 0.75 | *calculé ci-dessus* |
| $R_V$ | $\langle (\delta V)^2 / (\sigma^2 V \delta t) \rangle$ | 1.0 | 0.40 | *calculé ci-dessus* |
| $R_{SV}$ | $\langle \delta S \delta V / (\rho \sigma S V \delta t) \rangle$ | 1.0 | 0.60 | *calculé ci-dessus* |

**Conclusion de Bergomi :** $\sigma_{\text{réalisée}} / \sigma_{\text{implicite}} \approx \sqrt{R_V} \approx 0.63$, soit une **surestimation de la vol-de-vol d'environ 40%** par les smiles de marché. La corrélation $\rho$, elle, est bien capturée.

---
## Section 6 — Forward smiles & forward-start options

Un **forward call** de moneyness $\xi$ et de durée $\theta$ se déclenchant à $T_1$ paie :
$$\left(\frac{S_{T_1+\theta}}{S_{T_1}} - \xi\right)^+$$

On obtient le **forward smile** $\hat{\sigma}(\xi)$ en inversant le prix BS du forward call.

**Propriétés attendues selon Bergomi :**
- Les forward smiles sont **plus convexes** que le smile actuel (incertitude sur les vols futures)
- Dans Heston, ils convergent vers une courbe stationnaire pour $T_1 \gg 1/k$
- La convexité est **asymétrique** : plus forte pour $\xi > 1$ (skew inversement proportionnel à la vol)

In [ ]:
# ============================================================
#  FORWARD SMILE PAR SIMULATION MONTE CARLO (Heston)
# ============================================================
def simulate_heston_paths(S0, V0_init, V0, kappa, sigma, rho,
                           T, n_steps, n_paths, r=0.0, q=0.0, seed=42):
    """
    Simulation Euler-Maruyama du modèle de Heston.
    Retourne S(T) et V(T).
    """
    rng  = np.random.default_rng(seed)
    dt   = T / n_steps
    S    = np.full(n_paths, S0, dtype=float)
    V    = np.full(n_paths, V0_init, dtype=float)

    sqrt_dt = np.sqrt(dt)
    sqrt_1mr2 = np.sqrt(1 - rho**2)

    for _ in range(n_steps):
        Z1 = rng.standard_normal(n_paths)
        Z2 = rng.standard_normal(n_paths)
        Zw = rho * Z1 + sqrt_1mr2 * Z2

        V_pos = np.maximum(V, 0)
        sv    = np.sqrt(V_pos)
        V     = V + kappa * (V0 - V) * dt + sigma * sv * Zw * sqrt_dt
        V     = np.maximum(V, 0)  # réflexion en 0
        S     = S * np.exp((r - q - 0.5 * V_pos) * dt + sv * Z1 * sqrt_dt)

    return S, V


def forward_smile_heston(V_init, V0, kappa, sigma, rho,
                          T1, theta, xi_grid,
                          n_paths=40_000, n_steps_per_year=252):
    """
    Calcule le forward smile pour différentes maturités T1.
    """
    S0 = 100.0
    # Étape 1 : simuler jusqu'à T1
    n1 = max(1, int(T1 * n_steps_per_year))
    S1, V1 = simulate_heston_paths(S0, V_init, V0, kappa, sigma, rho,
                                    T=T1, n_steps=n1, n_paths=n_paths)

    # Étape 2 : pour chaque chemin, simuler encore theta
    n2 = max(1, int(theta * n_steps_per_year))
    S2, _ = simulate_heston_paths(1.0, V1.mean(), V0, kappa, sigma, rho,
                                   T=theta, n_steps=n2, n_paths=n_paths,
                                   seed=99)
    # Returns: S(T1+theta)/S(T1) — on approche grossièrement ici
    # (une simulation "path-by-path" serait exacte mais très lente)
    perf = S2  # normalisé à 1

    ivs = []
    for xi in xi_grid:
        payoff = np.maximum(perf - xi, 0.0)
        price  = payoff.mean()  # r=q=0
        if price < 1e-10:
            ivs.append(np.nan)
            continue
        iv = implied_vol(1.0, xi, theta, price, 0, 0, 'call')
        ivs.append(iv)
    return np.array(ivs)


# Paramètres de référence Bergomi
V_ref   = 0.10
V0_ref  = 0.10
kap_ref = 2.0
sig_ref = 1.0
rho_ref = -0.7

xi_1y = np.array([0.70, 0.80, 0.90, 1.00, 1.10, 1.20, 1.30, 1.40, 1.50, 1.60])
xi_3m = np.array([0.80, 0.85, 0.90, 0.95, 1.00, 1.05, 1.10, 1.15, 1.20, 1.25, 1.30])

T1_vals = [0.0, 0.25, 0.5, 1.0]

print('Calcul des forward smiles (MC Heston)...')
fs_1y = {}
fs_3m = {}
for T1 in T1_vals:
    fs_1y[T1] = forward_smile_heston(V_ref, V0_ref, kap_ref, sig_ref, rho_ref,
                                      T1, 1.0, xi_1y)
    fs_3m[T1] = forward_smile_heston(V_ref, V0_ref, kap_ref, sig_ref, rho_ref,
                                      T1, 0.25, xi_3m)
    print(f'  T1 = {T1}y terminé')
print('Done.')

In [ ]:
# ============================================================
#  FIGURE 3.4 — RÉPLIQUE FORWARD SMILES
# ============================================================
colors = ['k', 'steelblue', 'darkorange', 'forestgreen']
labels = ['Aujourd\'hui (T₁=0)', 'T₁ = 3 mois', 'T₁ = 6 mois', 'T₁ = 1 an']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for (T1, iv), color, label in zip(fs_1y.items(), colors, labels):
    axes[0].plot(xi_1y * 100, np.where(np.isfinite(iv), iv * 100, np.nan),
                 color=color, lw=2, label=label)
axes[0].set_xlabel('Moneyness ξ (%)')
axes[0].set_ylabel('Vol implicite (%)')
axes[0].set_title('Forward smile — maturité θ = 1 an')
axes[0].legend()

for (T1, iv), color, label in zip(fs_3m.items(), colors, labels):
    axes[1].plot(xi_3m * 100, np.where(np.isfinite(iv), iv * 100, np.nan),
                 color=color, lw=2, label=label)
axes[1].set_xlabel('Moneyness ξ (%)')
axes[1].set_ylabel('Vol implicite (%)')
axes[1].set_title('Forward smile — maturité θ = 3 mois')
axes[1].legend()

plt.suptitle('Figure 3.4 — Forward smiles Heston (réplique Bergomi)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
#  DENSITÉ STATIONNAIRE DE V  (Bergomi section 3.3)
# ============================================================
# ρ(V) ∝ V^(2kV0/σ² - 1) * exp(-2k/σ² * V)
# Loi Gamma renversée — Loi de Gamma

def stationary_density_V(V_grid, V0, kappa, sigma):
    alpha = 2 * kappa * V0 / sigma**2
    beta  = 2 * kappa / sigma**2
    log_dens = (alpha - 1) * np.log(V_grid) - beta * V_grid
    dens = np.exp(log_dens - log_dens.max())
    return dens / np.trapz(dens, V_grid)

alpha_ref = 2 * kap_ref * V0_ref / sig_ref**2
print(f'Paramètre alpha = 2kV₀/σ² - 1 = {alpha_ref - 1:.3f}')
print(f'→ densité {"diverge" if alpha_ref < 1 else "finie"} en V=0')

V_grid = np.linspace(0.001, 0.5, 500)
density = stationary_density_V(V_grid, V0_ref, kap_ref, sig_ref)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(V_grid * 100, density, 'steelblue', lw=2)
ax.axvline(V0_ref * 100, ls='--', color='firebrick', label=f'$V_0$ = {V0_ref*100:.0f}%')
ax.set_xlabel('V (variance instantanée, %)')
ax.set_ylabel('Densité stationnaire')
ax.set_title(f'Densité stationnaire de V — α = 2kV₀/σ² = {alpha_ref:.2f}')
ax.legend()
plt.tight_layout()
plt.show()

print('\nSi α < 1 (ici α = {:.2f}), la densité diverge en V=0 :'.format(alpha_ref))
print('les scénarios "vol basse / skew élevé" sont surpondérés par Heston.')

---
## Section 7 — Delta et comparaison avec le modèle Local Vol

### 7.1 Dynamique locale des vols implicites

**Local Vol (Dupire) :** À court terme et en limite de faible skew :
$$\frac{d\hat{\sigma}_{K=S}}{d\ln S} = 2\left.\frac{d\hat{\sigma}}{d\ln K}\right|_{K=S}$$
La vol ATM se déplace **deux fois plus vite** que le skew.

**Heston :** Conditionnellement à $\delta S$, on a $\mathbb{E}[\delta V] = \rho\sigma V/S \cdot \delta S$, donc :

- *Court terme* ($T \ll \tau$) : $\mathbb{E}[\delta\hat{\sigma}_{K=S}] / \delta\ln S = 2\, d\hat{\sigma}/d\ln K|_{K=S}$ → **identique à Local Vol**
- *Long terme* ($T \gg \tau$) : $\mathbb{E}[\delta\hat{\sigma}_{K=F}] / \delta\ln S = d\hat{\sigma}/d\ln K|_{K=F}$ → **sticky-strike** (vols fixes)

### 7.2 Comportement du Delta

In [ ]:
# ============================================================
#  DELTA HESTON vs BS vs LOCAL VOL — ILLUSTRATION
# ============================================================
def heston_delta_numerical(S0, K, T, V, V0, kappa, sigma, rho, r=0., q=0., eps=0.5):
    """Delta Heston par différence finie sur S0."""
    pu = heston_price(S0 + eps, K, T, V, V0, kappa, sigma, rho, r, q)
    pd = heston_price(S0 - eps, K, T, V, V0, kappa, sigma, rho, r, q)
    return (pu - pd) / (2 * eps)


S0 = 100.0
T_vals_delta = [0.1, 0.5, 1.0]
K_grid = np.linspace(70, 130, 30)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, T in zip(axes, T_vals_delta):
    delta_heston_list = []
    delta_bs_list     = []

    # Vol ATM Heston pour ce T (approximation court terme)
    sigma_atm = np.sqrt(V_ref)  # approximation

    for K in K_grid:
        # Delta Heston
        dh = heston_delta_numerical(S0, K, T, V_ref, V0_ref, kap_ref, sig_ref, rho_ref)
        delta_heston_list.append(dh)
        # Delta BS avec vol implicite de marché (approximation)
        lm = np.log(K / S0)
        iv_approx = sigma_atm + rho_ref * sig_ref / (4 * sigma_atm) * lm
        iv_approx = max(iv_approx, 0.01)
        px_bs = bs_price(S0, K, T, iv_approx)
        d1 = (np.log(S0 / K) + 0.5 * iv_approx**2 * T) / (iv_approx * np.sqrt(T))
        delta_bs_list.append(norm.cdf(d1))

    ax.plot(K_grid, delta_heston_list, 'steelblue', lw=2, label='Delta Heston')
    ax.plot(K_grid, delta_bs_list,     'firebrick',  lw=2, ls='--', label='Delta BS (vol impl)')
    ax.axvline(S0, ls=':', color='gray', lw=1)
    ax.set_xlabel('Strike K')
    ax.set_ylabel('Delta')
    ax.set_title(f'T = {T}y')
    ax.legend(fontsize=9)

plt.suptitle('Figure 3.4b — Delta Heston vs Black-Scholes pour différentes maturités',
             fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
#  DYNAMIQUE STICKY-STRIKE vs STICKY-DELTA — ILLUSTRATION
# ============================================================
# Bergomi montre que Heston passe de "like local vol" (CT) à "sticky strike" (LT)
# On illustre cela en choquant le spot de +5% et en regardant le déplacement du smile

def compute_smile(S0, T, V, V0, kappa, sigma, rho, log_m_range=(-0.3, 0.3), n=30):
    log_m = np.linspace(*log_m_range, n)
    K_vec = S0 * np.exp(log_m)
    ivs   = []
    for K in K_vec:
        px = heston_price(S0, K, T, V, V0, kappa, sigma, rho)
        if K < S0:
            px = px + S0 - K  # put-call parity
        iv = implied_vol(S0, K, T, max(px, 1e-10))
        ivs.append(iv)
    return K_vec, np.array(ivs)


S_base  = 100.0
S_shock = 105.0
shock   = (S_shock - S_base) / S_base  # +5%

for T_demo, label_T in [(0.1, 'court terme T=0.1y'), (1.5, 'long terme T=1.5y')]:
    K_base,  iv_base  = compute_smile(S_base,  T_demo, V_ref, V0_ref, kap_ref, sig_ref, rho_ref)
    K_shock, iv_shock = compute_smile(S_shock, T_demo, V_ref, V0_ref, kap_ref, sig_ref, rho_ref)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(K_base,  np.where(np.isfinite(iv_base),  iv_base  * 100, np.nan),
            'steelblue', lw=2, label=f'S₀ = {S_base}')
    ax.plot(K_shock, np.where(np.isfinite(iv_shock), iv_shock * 100, np.nan),
            'firebrick', lw=2, ls='--', label=f'S₀ = {S_shock} (+5%)')
    ax.set_xlabel('Strike K')
    ax.set_ylabel('Vol implicite (%)')
    ax.set_title(f'Déplacement du smile après choc spot de +5% — {label_T}\n'
                 f'(CT ≈ Local Vol ; LT ≈ Sticky Strike)')
    ax.legend()
    plt.tight_layout()
    plt.show()

---
## Section 8 — Modèles Jump/Lévy : Variance Swaps

### 8.1 Propriétés structurelles des modèles Jump/Lévy

Dans les modèles de Lévy homogènes :
- Les smiles se **translatent** avec le spot (moneyness fixe → vol fixe)
- Les forward smiles sont **identiques** au smile actuel
- Le skew décroît en $1/T$ (trop vite vs marché)

### 8.2 Variance Swap vs Log Swap

Le **Log Swap** réplique la variance réalisée par delta-hedging de $-2\ln(S)$. Son P&L de Gamma à l'ordre 3 vaut :
$$\mathbb{E}\left[\left(\frac{\Delta S}{S}\right)^2 - \frac{2}{3}\left(\frac{\Delta S}{S}\right)^3\right] \approx \sigma^2 \Delta t \left(1 - \frac{2\mathcal{S}_{\Delta t}}{3}\sigma\sqrt{\Delta t}\right)$$

où $\mathcal{S}_{\Delta t}$ est le **skewness** des rendements.

- **Diffusion (Heston)** : $\mathcal{S}_{\Delta t} \to 0$ quand $\Delta t \to 0$ → $\hat{\sigma}_{VS} = \hat{\sigma}_{LS}$
- **Lévy** : $\mathcal{S}_{\Delta t} \propto 1/\sqrt{\Delta t}$ → le terme du 3e ordre reste fini → $\hat{\sigma}_{VS} < \hat{\sigma}_{LS}$

**Relation de Bergomi (1er ordre en skewness) :**
$$\hat{\sigma}_{K=F} - \hat{\sigma}_{VS} = 3(\hat{\sigma}_{LS} - \hat{\sigma}_{K=F})$$

In [ ]:
# ============================================================
#  CALCUL DE sigma_LS (Log Swap vol) DEPUIS LES DONNÉES DE MARCHÉ
#  sigma_LS^2 = 2/T * [integrale sur K des puts/calls / K^2 dk]
# ============================================================
def logswap_vol_from_smile(strikes, ivs, S, T, r=0.0, q=0.0):
    """
    Réplication statique du Log Swap :
    σ²_LS * T = 2 * [∫_0^F Put(K)/K² dK + ∫_F^∞ Call(K)/K² dK]
    """
    F = S * np.exp((r - q) * T)
    mask = np.isfinite(ivs) & (ivs > 0)
    K_arr = strikes[mask]
    iv_arr = ivs[mask]

    if len(K_arr) < 3:
        return np.nan

    prices_arr = np.array([
        bs_price(S, K, T, iv, r, q, 'put' if K <= F else 'call')
        for K, iv in zip(K_arr, iv_arr)
    ])
    integrand = 2 * prices_arr / K_arr**2
    integral  = np.trapz(integrand, K_arr)
    vs2 = integral / T
    return np.sqrt(max(vs2, 0))


def atmf_vol_and_ls_by_date(surface, target_T=1.0, tol=0.05):
    rows = []
    for date, grp in surface.groupby('date'):
        avail_T = grp['T'].unique()
        closest = avail_T[np.argmin(np.abs(avail_T - target_T))]
        if abs(closest - target_T) > tol:
            continue
        sl = grp[np.abs(grp['T'] - closest) < 1e-3].copy()
        sl = sl.sort_values('strike')
        S  = sl['spot'].iloc[0]
        # ATM vol
        idx_atm = sl['log_moneyness'].abs().idxmin()
        atm_iv  = sl.loc[idx_atm, 'market_iv']
        # Log Swap vol
        ls_vol  = logswap_vol_from_smile(
            sl['strike'].values, sl['market_iv'].values, S, closest
        )
        rows.append({'date': date, 'atm_iv': atm_iv, 'ls_vol': ls_vol,
                     'spread_bps': (ls_vol - atm_iv) * 10_000 if ls_vol else np.nan})
    df = pd.DataFrame(rows).set_index('date')
    df.index = pd.to_datetime(df.index)
    return df.sort_index()


ls_df = atmf_vol_and_ls_by_date(surface, target_T=1.0)
print(f'{len(ls_df)} dates traitées')
print('\nStatistiques spread σ_LS - σ_ATM (bps) :')
ls_df['spread_bps'].describe().round(1)

In [ ]:
# ============================================================
#  FIGURE 8 — σ_ATM vs σ_LS (1 an) — analogue exemple Bergomi
# ============================================================
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

axes[0].plot(ls_df.index, ls_df['atm_iv'] * 100, 'steelblue', lw=1.5, label='σ_ATM (1Y)')
axes[0].plot(ls_df.index, ls_df['ls_vol'] * 100,  'firebrick', lw=1.5, ls='--',
             label='σ_LS (Log Swap, 1Y)')
axes[0].set_ylabel('Volatilité (%)')
axes[0].set_title('Vol ATM 1 an vs Log Swap Vol (SX5E)')
axes[0].legend()
axes[0].xaxis.set_tick_params(rotation=30)

axes[1].fill_between(ls_df.index, ls_df['spread_bps'].fillna(0),
                     alpha=0.6, color='darkorange', label='σ_LS - σ_ATM (bps)')
axes[1].axhline(ls_df['spread_bps'].median(), ls='--', color='k',
                label=f'Médiane = {ls_df["spread_bps"].median():.0f} bps')
axes[1].set_ylabel('Spread (bps)')
axes[1].set_title('Écart σ_LS − σ_ATM — Bergomi : ~400 bps en mars 2004')
axes[1].legend()
axes[1].xaxis.set_tick_params(rotation=30)

plt.suptitle('Figure 8 — Variance Swap / Log Swap (Section 4.2 Bergomi)', fontweight='bold')
plt.tight_layout()
plt.show()

print('\n→ La différence σ_LS - σ_ATM est d\'autant plus grande que le skew est élevé.')
print('  Dans les modèles diffusifs (Heston), σ_VS = σ_LS par construction.')
print('  Dans les modèles Lévy, σ_VS < σ_LS (correction du terme d\'ordre 3).')

In [ ]:
# ============================================================
#  SKEWNESS DES RENDEMENTS JOURNALIERS SX5E
#  (Bergomi section 4.2 : S_Δt est d'ordre 1 empiriquement)
# ============================================================
from scipy.stats import skew as scipy_skew, kurtosis as scipy_kurt

spot_series_full = (
    surface.drop_duplicates('date')[['date', 'spot']]
    .set_index('date').sort_index()
)
spot_series_full.index = pd.to_datetime(spot_series_full.index)

log_ret = np.log(spot_series_full['spot']).diff().dropna()

sk   = scipy_skew(log_ret)
kurt = scipy_kurt(log_ret, fisher=True)  # excès de kurtosis
vol_daily = log_ret.std()

# Estimation du terme correctif pour VS pricing
# P&L normalisé : 1 - (2/3) * S_dt * sigma * sqrt(dt)
dt_year = 1 / 252.0
correction = 2 / 3 * abs(sk) * vol_daily * np.sqrt(dt_year) / dt_year

print('=' * 60)
print('  Statistiques des rendements journaliers SX5E')
print('=' * 60)
print(f'  N observations       : {len(log_ret)}')
print(f'  Vol annualisée       : {vol_daily * np.sqrt(252) * 100:.2f}%')
print(f'  Skewness journalier  : {sk:.4f}   (Bergomi : ~ -1 à -0.5)')
print(f'  Kurtosis excess      : {kurt:.4f}')
print()
print(f'  Contribution terme ordre 3 (Lévy) : {correction:.4f}')
print(f'  Contribution terme ordre 2        : 1.0000')
print(f'  Ratio 3rd/2nd order               : {correction:.4f}')
print(f'  → Bergomi : ~50 fois plus petit en empirique vs modèle Lévy')
print('=' * 60)

In [ ]:
# ============================================================
#  DISTRIBUTION DES RENDEMENTS vs NORMALE
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogramme vs normale
ret_arr = log_ret.values
x_grid  = np.linspace(ret_arr.min(), ret_arr.max(), 200)
pdf_norm = norm.pdf(x_grid, ret_arr.mean(), ret_arr.std())

axes[0].hist(ret_arr, bins=100, density=True, color='steelblue',
             alpha=0.7, label='Rendements journaliers SX5E')
axes[0].plot(x_grid, pdf_norm, 'firebrick', lw=2, label='Normale')
axes[0].set_xlabel('Rendement log-journalier')
axes[0].set_ylabel('Densité')
axes[0].set_title(f'Distribution des rendements\n(skew={sk:.3f}, kurt={kurt:.3f})')
axes[0].legend()

# QQ-Plot
from scipy.stats import probplot
probplot(ret_arr, dist='norm', plot=axes[1])
axes[1].set_title('QQ-Plot vs Normale')
axes[1].get_lines()[1].set_color('firebrick')

plt.suptitle('Section 4.2 — Propriétés empiriques des rendements SX5E', fontweight='bold')
plt.tight_layout()
plt.show()

---
## Section 9 — Extension stochastique des modèles de Lévy

### 9.1 Subordination stochastique (Carr et al., 2003)

On remplace $t$ par un processus de temps stochastique $\tau_t = \int_0^t \lambda_u\,du$ :
$$d\lambda = -k(\lambda - \lambda_0)\,dt + \sigma\sqrt{\lambda}\,dZ_t$$

Pour les options à court terme ($\lambda$ agit comme facteur d'échelle) :
$$\hat{\sigma}_{K=F} \propto \sqrt{\lambda}, \qquad \left.K\frac{d\hat{\sigma}}{dK}\right|_{K=F} \propto \frac{1}{\sqrt{\lambda T}}$$

**Résultat structurel :**
$$\left.K\frac{d\hat{\sigma}}{dK}\right|_{K=F} \propto \frac{1}{\hat{\sigma}_{K=F}}$$

Le skew est **inversement proportionnel à la vol ATM** — tout comme dans le modèle de Heston ! C'est une propriété structurelle de **toute la classe** des modèles de Lévy à temps stochastique.

In [ ]:
# ============================================================
#  VÉRIFICATION EMPIRIQUE : skew ∝ 1/vol_ATM ?
# ============================================================
def compute_skew_and_atm(surface, T_target=1/12, tol=0.02):
    """Calcule le skew ≈ (σ(90%) - σ(110%)) / 20 ln-strikes et la vol ATM."""
    rows = []
    for date, grp in surface.groupby('date'):
        avail = grp['T'].unique()
        closest = avail[np.argmin(np.abs(avail - T_target))]
        if abs(closest - T_target) > tol:
            continue
        sl = grp[np.abs(grp['T'] - closest) < 1e-3].copy()
        sl = sl.sort_values('log_moneyness')
        # ATM
        if sl['log_moneyness'].abs().min() > 0.05:
            continue
        atm_iv = sl.loc[sl['log_moneyness'].abs().idxmin(), 'market_iv']
        # Skew : pente de la vol entre log_moneyness ~ -0.1 et +0.1
        sl_sub = sl[sl['log_moneyness'].abs() < 0.15]
        if len(sl_sub) < 3:
            continue
        coef = np.polyfit(sl_sub['log_moneyness'], sl_sub['market_iv'], 1)
        skew_slope = coef[0]  # d_iv / d_ln_K
        rows.append({'date': date, 'atm_iv': atm_iv, 'skew': skew_slope})

    df = pd.DataFrame(rows).set_index('date')
    df.index = pd.to_datetime(df.index)
    return df.sort_index().dropna()


skew_df = compute_skew_and_atm(surface, T_target=1/12, tol=0.02)

if len(skew_df) > 10:
    corr_sv = skew_df['skew'].corr(skew_df['atm_iv'])
    corr_inv = skew_df['skew'].corr(1 / skew_df['atm_iv'])

    print(f'Corrélation(skew, vol_ATM)     = {corr_sv:.3f}')
    print(f'Corrélation(skew, 1/vol_ATM)   = {corr_inv:.3f}')
    print()

    if corr_sv > 0:
        print('→ Skew proportionnel à la vol ATM (skew augmente avec la vol)')
        print('  C\'est le comportement EMPIRIQUE (fonds vol, Bergomi section 3.2.1)')
    else:
        print('→ Skew inversement proportionnel à la vol ATM')
        print('  C\'est le comportement Heston / Lévy stochastique')

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].scatter(skew_df['atm_iv'] * 100, skew_df['skew'] * 100,
                    alpha=0.3, s=15, color='steelblue')
    axes[0].set_xlabel('Vol ATM 1M (%)')
    axes[0].set_ylabel('Skew (d_iv / d_ln_K)')
    axes[0].set_title(f'Skew vs Vol ATM  —  corr = {corr_sv:.3f}')

    axes[1].scatter(1 / skew_df['atm_iv'], skew_df['skew'] * 100,
                    alpha=0.3, s=15, color='firebrick')
    axes[1].set_xlabel('1 / Vol ATM')
    axes[1].set_ylabel('Skew (d_iv / d_ln_K)')
    axes[1].set_title(f'Skew vs 1/Vol ATM  —  corr = {corr_inv:.3f}\n'
                      f'(Heston / Lévy-stochastique prédisent skew ∝ 1/σ_ATM)')

    plt.suptitle('Section 9 — Test structurel : skew ∝ vol_ATM ou 1/vol_ATM ?',
                 fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print(f'Pas assez de données pour le test ({len(skew_df)} dates). Vérifier la surface.')

In [ ]:
# ============================================================
#  STRUCTURE PAR TERME DU SKEW — décroissance 1/T vs 1/sqrt(T)
# ============================================================
def skew_by_maturity(surface):
    """Calcule le skew moyen pour chaque maturité disponible."""
    rows = []
    for (date, T), grp in surface.groupby(['date', 'T']):
        sl = grp.sort_values('log_moneyness')
        if len(sl) < 4:
            continue
        sl_sub = sl[sl['log_moneyness'].abs() < 0.2]
        if len(sl_sub) < 3:
            continue
        coef = np.polyfit(sl_sub['log_moneyness'], sl_sub['market_iv'], 1)
        rows.append({'date': date, 'T': T, 'skew': coef[0]})

    df = pd.DataFrame(rows)
    avg = df.groupby('T')['skew'].median().reset_index()
    avg.columns = ['T', 'median_skew']
    return avg.sort_values('T')


skew_struct = skew_by_maturity(surface)

T_arr   = skew_struct['T'].values
sk_arr  = np.abs(skew_struct['median_skew'].values)

# Fit 1/T et 1/sqrt(T)
T_fit = T_arr[T_arr > 0.05]
sk_fit = sk_arr[:len(T_fit)]

c1, _ = np.polyfit(np.log(T_fit), np.log(sk_fit + 1e-10), 1, cov=True)[:2]
slope = c1[0]

T_plot = np.linspace(T_arr.min(), T_arr.max(), 100)

fig, ax = plt.subplots(figsize=(10, 5))
ax.loglog(T_arr, sk_arr, 'o', color='steelblue', ms=6, label='Skew médian (marché)')

# Normalisation au premier point
idx0 = np.argmin(np.abs(T_arr - 0.1))
ref  = sk_arr[idx0] * T_arr[idx0]

ax.loglog(T_plot, ref / T_plot,       'firebrick', ls='--', lw=2, label='∝ 1/T  (Lévy / Heston LT)')
ax.loglog(T_plot, ref / np.sqrt(T_plot) * np.sqrt(T_arr[idx0]),
           'darkorange', ls='-.', lw=2, label='∝ 1/√T  (stochastique vol pur)')
ax.loglog(T_plot, ref / T_plot**abs(slope) * T_arr[idx0]**abs(slope),
           'forestgreen', ls=':', lw=2, label=f'Fit empirique ∝ 1/T^{abs(slope):.2f}')

ax.set_xlabel('Maturité T (années)')
ax.set_ylabel('|Skew| = |d_iv / d_ln_K|')
ax.set_title('Structure par terme du skew — SX5E')
ax.legend()
plt.tight_layout()
plt.show()

print(f'\nPente du skew empirique (log-log) : T^{slope:.3f}')
print('Théorie Lévy / Heston LT : -1.00')
print('Théorie Heston CT : -0.50')

---
## Section 10 — Synthèse et conclusions

### 10.1 Résumé des résultats empiriques

In [ ]:
# ============================================================
#  TABLEAU DE SYNTHÈSE — RÉPLIQUE DES RÉSULTATS BERGOMI
# ============================================================
results_table = pd.DataFrame([
    {
        'Métrique': 'R_S (variance spot)',
        'Bergomi (1999-2004)': 0.75,
        'Notre estimation (5 ans)': round(R_S, 3),
        'Valeur théorique': 1.0,
        'Interprétation': 'Vols ST surestiment la vol historique'
    },
    {
        'Métrique': 'R_V (variance vol-de-vol)',
        'Bergomi (1999-2004)': 0.40,
        'Notre estimation (5 ans)': round(R_V, 3),
        'Valeur théorique': 1.0,
        'Interprétation': 'Vol-de-vol σ surestimée ~2x par smiles'
    },
    {
        'Métrique': 'R_SV (covariance spot-vol)',
        'Bergomi (1999-2004)': 0.60,
        'Notre estimation (5 ans)': round(R_SV, 3),
        'Valeur théorique': 1.0,
        'Interprétation': 'Corrélation ρ bien capturée'
    },
    {
        'Métrique': 'σ_réal / σ_impl',
        'Bergomi (1999-2004)': 0.63,
        'Notre estimation (5 ans)': round(np.sqrt(R_V), 3),
        'Valeur théorique': 1.0,
        'Interprétation': 'Biais de rehedging Vega non pricé'
    },
    {
        'Métrique': 'ρ_réal / ρ_impl',
        'Bergomi (1999-2004)': 1.10,
        'Notre estimation (5 ans)': round(R_SV / (np.sqrt(R_V * R_S)), 3),
        'Valeur théorique': 1.0,
        'Interprétation': 'Delta vanna efficace'
    },
])

results_table.set_index('Métrique')

In [ ]:
# ============================================================
#  FIGURE SYNTHÈSE FINALE — Dashboard Bergomi complet
# ============================================================
fig = plt.figure(figsize=(18, 14))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

# 1. Vol ATM 1M vs sqrt(V)
ax1 = fig.add_subplot(gs[0, 0])
c1m = atm_vols['1M'].join(calib_df[['V']], how='inner')
ax1.plot(c1m.index, c1m['atm_iv'] * 100, 'steelblue', lw=1, label='ATM 1M')
ax1.plot(c1m.index, np.sqrt(c1m['V']) * 100, 'firebrick', lw=1, ls='--', label='√V')
ax1.set_title('Vol ATM 1M vs √V')
ax1.legend(fontsize=8)
ax1.xaxis.set_tick_params(rotation=30, labelsize=7)

# 2. Paramètre V
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(calib_df.index, calib_df['V'] * 100, 'steelblue', lw=1)
ax2.set_title('V (variance instantanée, %²)')
ax2.xaxis.set_tick_params(rotation=30, labelsize=7)

# 3. Paramètre sigma
ax3 = fig.add_subplot(gs[0, 2])
ax3.plot(calib_df.index, calib_df['sigma'], 'forestgreen', lw=1)
ax3.set_title('σ (vol-de-vol)')
ax3.xaxis.set_tick_params(rotation=30, labelsize=7)

# 4. Paramètre rho
ax4 = fig.add_subplot(gs[1, 0])
ax4.plot(calib_df.index, calib_df['rho'], 'firebrick', lw=1)
ax4.set_title('ρ (corrélation spot/vol)')
ax4.xaxis.set_tick_params(rotation=30, labelsize=7)

# 5. Structure par terme du skew
ax5 = fig.add_subplot(gs[1, 1])
ax5.loglog(skew_struct['T'], np.abs(skew_struct['median_skew']), 'o-',
            color='steelblue', ms=4, lw=1.5)
T_ref_line = np.linspace(skew_struct['T'].min(), skew_struct['T'].max(), 50)
ax5.loglog(T_ref_line, 0.05 / T_ref_line, 'firebrick', ls='--', lw=1.5, label='1/T')
ax5.set_xlabel('T')
ax5.set_title('Structure terme skew')
ax5.legend(fontsize=8)

# 6. σ_ATM vs σ_LS
ax6 = fig.add_subplot(gs[1, 2])
ax6.plot(ls_df.index, ls_df['atm_iv'] * 100, 'steelblue', lw=1, label='σ_ATM')
ax6.plot(ls_df.index, ls_df['ls_vol']  * 100, 'firebrick', lw=1, ls='--', label='σ_LS')
ax6.set_title('σ_ATM vs σ_LS (1Y)')
ax6.legend(fontsize=8)
ax6.xaxis.set_tick_params(rotation=30, labelsize=7)

# 7. Corrélation sigma ~ V
ax7 = fig.add_subplot(gs[2, 0])
ax7.scatter(calib_df['V'] * 100, calib_df['sigma'], alpha=0.3, s=8, color='darkorange')
ax7.set_xlabel('V (%²)')
ax7.set_ylabel('σ')
ax7.set_title(f'σ vs V — corr={corr_sigma_V:.2f}')

# 8. RMSE de calibration
ax8 = fig.add_subplot(gs[2, 1])
ax8.plot(calib_df.index, calib_df['rmse'] * 10_000, 'purple', lw=1)
ax8.axhline(calib_df['rmse'].median() * 10_000, ls='--', color='k',
             label=f'Médiane={calib_df["rmse"].median()*10000:.0f} bps')
ax8.set_title('RMSE calibration Heston (bps)')
ax8.legend(fontsize=8)
ax8.xaxis.set_tick_params(rotation=30, labelsize=7)

# 9. Distribution rendements
ax9 = fig.add_subplot(gs[2, 2])
ax9.hist(log_ret.values * 100, bins=80, density=True, color='steelblue', alpha=0.7)
x_r = np.linspace(log_ret.min() * 100, log_ret.max() * 100, 200)
ax9.plot(x_r, norm.pdf(x_r, log_ret.mean() * 100, log_ret.std() * 100),
         'firebrick', lw=2)
ax9.set_title(f'Rendements SX5E (sk={sk:.2f})')
ax9.set_xlabel('Rendement journalier (%)')

plt.suptitle('Dashboard Bergomi (2004) — SX5E, 5 ans glissants',
             fontsize=15, fontweight='bold', y=1.01)
plt.savefig('bergomi_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Dashboard sauvegardé : bergomi_dashboard.png')

---
## Section 10.2 — Conclusions de Bergomi & implications pratiques

### Résultats structurels

| Classe de modèles | Décroissance du skew | Dynamique ATMF vol | σ_VS vs σ_LS | Limitation structurelle |
|---|---|---|---|---|
| **Heston (1 facteur)** | $1/T$ (LT) / $1/\sqrt{T}$ (CT) | Sticky-strike (LT) / Local-vol-like (CT) | $=$ | Vol-de-vol σ surestimée ×2 |
| **Jump/Lévy** | $1/T$ (trop vite) | Sticky-delta | $<$ | Skew excessif aux courtes échéances |
| **Lévy stochastique** | Dépend de λ | Intermédiaire | $<$ | Skew ∝ 1/σ_ATM (rigide) |

### Message principal

> *"En plus du processus spot, au moins un autre processus driving est nécessaire pour modéliser la dynamique des vols implicites — et probablement plus d'un si l'objectif est de capturer correctement la structure par terme du 'vol des vols'."*
> — **Bergomi (2004), conclusion**

Cette observation est à l'origine du développement des **modèles de variance forward stochastique** (Bergomi 2005, 2008), où la surface de variance forward entière devient le processus d'état.

In [ ]:
# ============================================================
#  BILAN QUANTITATIF FINAL
# ============================================================
print('=' * 70)
print('  BILAN — Smile Dynamics (Bergomi 2004) sur SX5E, 5 ans')
print('=' * 70)

print(f'''
  DONNÉES
  ─────────────────────────────────────────────────────────────
  Période      : {date_start.date()} → {date_end.date()}
  Points surf. : {surface.shape[0]:,}
  Dates calib. : {len(calib_df)}

  CALIBRATION HESTON (k = {KAPPA_FIXED})
  ─────────────────────────────────────────────────────────────
  V    médiane : {calib_df["V"].median()*100:.2f}%²
  V0   médiane : {calib_df["V0"].median()*100:.2f}%²
  σ    médiane : {calib_df["sigma"].median():.3f}
  ρ    médiane : {calib_df["rho"].median():.3f}
  RMSE médiane : {calib_df["rmse"].median()*10000:.1f} bps

  DYNAMIQUE — RATIOS DE CONSISTANCE
  ─────────────────────────────────────────────────────────────
  R_S   = {R_S:.3f}  (Bergomi : 0.75)  — surestimation vol historique : {(1-R_S)*100:.0f}%
  R_V   = {R_V:.3f}  (Bergomi : 0.40)  — σ impl / σ réal = {1/np.sqrt(R_V):.2f}x
  R_SV  = {R_SV:.3f}  (Bergomi : 0.60)  — ρ bien capturé

  VARIANCE SWAP / LOG SWAP
  ─────────────────────────────────────────────────────────────
  Spread médian σ_LS - σ_ATM (1Y) : {ls_df['spread_bps'].median():.0f} bps
  (Bergomi: ~400 bps en mars 2004)

  STRUCTURE PAR TERME DU SKEW
  ─────────────────────────────────────────────────────────────
  Pente log-log empirique : T^{slope:.2f}
  Lévy / Heston LT : T^-1.00
  Heston CT        : T^-0.50

  CONCLUSION
  ─────────────────────────────────────────────────────────────
  Le modèle de Heston à 1 facteur capte bien ρ mais surévalue σ
  d\'un facteur ~{1/np.sqrt(R_V):.1f}x. Un modèle à plusieurs facteurs de variance
  (type Bergomi 2005) est nécessaire pour le pricing correct
  d\'options path-dependent (Napoleons, reverse cliquets).
''')

print('=' * 70)

---

## Références

- **Bergomi, L. (2004)** — *Smile Dynamics*, Société Générale (SSRN 1493294)
- **Heston, S. (1993)** — *A closed-form solution for options with stochastic volatility*, Review of Financial Studies, 6, 327–343
- **Dupire, B. (1994)** — *Pricing with a smile*, Risk, January, 18–20
- **Carr, P., Geman, H., Madan, D., Yor, M. (2003)** — *Stochastic Volatility for Lévy Processes*, Mathematical Finance, 13(3), 345–382
- **Merton, R. (1976)** — *Option pricing with discontinuous returns*, Bell Journal of Financial Economics, 3, 145–166
- **Backus, D., Foresi, S., Li, K., Wu, L. (1997)** — *Accounting for biases in Black-Scholes*, unpublished

---
*Notebook réalisé pour l'implémentation complète de Bergomi (2004) sur données SX5E via MDX — Anthropic Claude*